# i01 · ¿Cuánto ha crecido la población de España y quién explica ese crecimiento?

- **Post:** `posts/i01-population-nationality/index.qmd`
- **Afirmación:** «España solo crece gracias a la inmigración.»
- **Pregunta a contrastar:** ¿cuánto ha crecido la población residente desde 2002 y qué parte de ese crecimiento corresponde a personas de nacionalidad extranjera o nacidas en el extranjero?
- **Datos:** INE (Cifras de Población hasta 2021, Estadística Continua de Población desde 2021)
  - descarga: `python src/download/download_ine_poblacion.py`
  - procesado: `python src/process/05_poblacion_nacionalidad.py`
  - salida: `data/processed/poblacion_nacionalidad.parquet`, `data/processed/poblacion_pais_nacimiento.parquet`

In [40]:
import os 
import polars as pl

project_path =  os.path.join(os.getcwd(), '..')  
raw_data_folder_path = os.path.join(project_path, 'data', 'raw', 'ine_poblacion')

## 1. Datos crudos

TODO: abrir los ficheros de `RAW / "ine_poblacion"` y entender su estructura antes de escribir `src/process/05_poblacion_nacionalidad.py`.

- ¿Qué series trae la tabla? ¿Totales y desgloses mezclados?
- ¿Fechas: 1 de enero, 1 de julio, trimestrales?
- ¿Hay valores provisionales?

In [41]:
raw_file_paths = {}
for file in os.listdir(raw_data_folder_path):
    raw_file_paths[file.split('.')[0]] = os.path.join(raw_data_folder_path, file)

In [42]:
raw_file_paths

{'56938': 'c:\\Users\\fscielzo\\Documents\\la-mirada-del-dato\\notebooks\\..\\data\\raw\\ine_poblacion\\56938.csv',
 '60129': 'c:\\Users\\fscielzo\\Documents\\la-mirada-del-dato\\notebooks\\..\\data\\raw\\ine_poblacion\\60129.csv',
 '60130': 'c:\\Users\\fscielzo\\Documents\\la-mirada-del-dato\\notebooks\\..\\data\\raw\\ine_poblacion\\60130.csv',
 '9691': 'c:\\Users\\fscielzo\\Documents\\la-mirada-del-dato\\notebooks\\..\\data\\raw\\ine_poblacion\\9691.csv'}

In [43]:
df = pl.read_csv(raw_file_paths['56938'], separator=';')
df

Sexo,Edad simple,Nacionalidad,País de nacimiento,Periodo,Total
str,str,str,str,str,str
"""Total""","""Todas las edades""","""Total""","""Total""","""1 de enero de 2025""","""49.128.297"""
"""Total""","""Todas las edades""","""Total""","""Total""","""1 de octubre de 2024""","""48.999.880"""
"""Total""","""Todas las edades""","""Total""","""Total""","""1 de julio de 2024""","""48.821.936"""
"""Total""","""Todas las edades""","""Total""","""Total""","""1 de abril de 2024""","""48.701.130"""
"""Total""","""Todas las edades""","""Total""","""Total""","""1 de enero de 2024""","""48.619.695"""
…,…,…,…,…,…
"""Mujeres""","""90 y más años""","""Extranjera""","""Extranjero""","""1 de enero de 2003""","""1.402"""
"""Mujeres""","""90 y más años""","""Extranjera""","""Extranjero""","""1 de octubre de 2002""",""""""
"""Mujeres""","""90 y más años""","""Extranjera""","""Extranjero""","""1 de julio de 2002""","""1.285"""


In [44]:
# Procesamiento 56938.csv

# Diccionario para mapear los meses en español a números
meses = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12"
}

# Transformación de la tabla
df = df.with_columns([
    # 1. Convertir 'Total' a numérico (Int64): elimina puntos de miles y maneja vacíos como null
    pl.col("Total")
      .str.replace_all(r"\.", "")
      .cast(pl.Int64, strict=False),

    # 2. Convertir 'Periodo' a fecha (pl.Date)
    pl.col("Periodo")
      .str.replace_many(list(meses.keys()), list(meses.values()))
      .str.to_date(format="%d de %m de %Y"),

    # 3. Extrae el primer número de la cadena y lo convierte a entero
    pl.col("Edad simple") # Ajusta el nombre si en tu df es "Edad"
      .str.extract(r"^(\d+)", 1)
      .cast(pl.Int8)
      .alias("edad_num")
      # Transformaciones resultantes:
      # "0 años" -> 0
      # "13 años" -> 13
      # "90 y más años" -> 90 (extrae el límite inferior numérico)
      # "Todas las edades" -> null (al no tener números, la extracción genera valor nulo)
])

df

Sexo,Edad simple,Nacionalidad,País de nacimiento,Periodo,Total,edad_num
str,str,str,str,date,i64,i8
"""Total""","""Todas las edades""","""Total""","""Total""",2025-01-01,49128297,null
"""Total""","""Todas las edades""","""Total""","""Total""",2024-10-01,48999880,null
"""Total""","""Todas las edades""","""Total""","""Total""",2024-07-01,48821936,null
"""Total""","""Todas las edades""","""Total""","""Total""",2024-04-01,48701130,null
"""Total""","""Todas las edades""","""Total""","""Total""",2024-01-01,48619695,null
…,…,…,…,…,…,…
"""Mujeres""","""90 y más años""","""Extranjera""","""Extranjero""",2003-01-01,1402,90
"""Mujeres""","""90 y más años""","""Extranjera""","""Extranjero""",2002-10-01,null,90
"""Mujeres""","""90 y más años""","""Extranjera""","""Extranjero""",2002-07-01,1285,90


In [45]:
df = pl.read_csv(raw_file_paths['60129'], separator=';')
df

Provincias,Nacionalidad,Grupo quinquenal de edad,Sexo,Periodo,Total
str,str,str,str,str,str
"""Total Nacional""","""Total""","""Todas las edades""","""Total""","""1 de julio de 2026""","""49.409.876"""
"""Total Nacional""","""Total""","""Todas las edades""","""Total""","""1 de abril de 2026""","""49.308.821"""
"""Total Nacional""","""Total""","""Todas las edades""","""Total""","""1 de enero de 2026""","""49.210.373"""
"""Total Nacional""","""Total""","""Todas las edades""","""Total""","""1 de octubre de 2025""","""49.117.283"""
"""Total Nacional""","""Total""","""Todas las edades""","""Total""","""1 de julio de 2025""","""48.978.224"""
…,…,…,…,…,…
"""52 Melilla""","""Extranjera""","""90 y más años""","""Mujeres""","""1 de enero de 2022""","""1"""
"""52 Melilla""","""Extranjera""","""90 y más años""","""Mujeres""","""1 de octubre de 2021""","""2"""
"""52 Melilla""","""Extranjera""","""90 y más años""","""Mujeres""","""1 de julio de 2021""","""4"""


In [46]:
df['Provincias'].unique().to_list()

['08 Barcelona',
 '16 Cuenca',
 '20 Gipuzkoa',
 '38 Santa Cruz de Tenerife',
 '25 Lleida',
 '32 Ourense',
 '37 Salamanca',
 '29 Málaga',
 '15 Coruña, A',
 '28 Madrid',
 '24 León',
 '13 Ciudad Real',
 '27 Lugo',
 '07 Balears, Illes',
 '33 Asturias',
 '11 Cádiz',
 '40 Segovia',
 '30 Murcia',
 '21 Huelva',
 '05 Ávila',
 '31 Navarra',
 '36 Pontevedra',
 '22 Huesca',
 '48 Bizkaia',
 '04 Almería',
 '49 Zamora',
 '45 Toledo',
 '35 Palmas, Las',
 '06 Badajoz',
 '42 Soria',
 '44 Teruel',
 '46 Valencia/València',
 '43 Tarragona',
 '41 Sevilla',
 '52 Melilla',
 '39 Cantabria',
 '19 Guadalajara',
 '14 Córdoba',
 '47 Valladolid',
 '26 Rioja, La',
 '09 Burgos',
 '02 Albacete',
 '51 Ceuta',
 '17 Girona',
 '10 Cáceres',
 '03 Alicante/Alacant',
 '23 Jaén',
 '18 Granada',
 '34 Palencia',
 '01 Araba/Álava',
 'Total Nacional',
 '12 Castellón/Castelló',
 '50 Zaragoza']

## 2. Comprobaciones

- [ ] Española + extranjera = total en cada fecha
- [ ] Nacida en España + nacida fuera = total
- [ ] Salto entre Cifras de Población y ECP en 2021: ¿se pueden enlazar?

## 3. Exploración

Ideas:
- Población total y variación anual
- Española vs extranjera; nacida en España vs fuera (niveles y % sobre el total)
- Descomposición: Δ total = Δ nacidos en España + Δ nacidos fuera, por periodos (2002–2008, 2008–2014, 2014–2019, 2019–hoy)
- Nacidos fuera por grandes zonas de origen

## 4. Cifras y gráficos finales (→ post)

## Notas

- Decisiones tomadas y por qué.
- Dudas abiertas / limitaciones.